FILE: flores_phage_pipeline.ipynb
==

In [ ]:
#============================================
#1 Imports
#============================================

import os
import shutil
import subprocess
from pathlib import Path

In [ ]:
#============================================
#2 Config Setup
#============================================

SRA_ID = "SRR37631993"
THREADS = 2

# adapter auto-detect by fastp
MIN_LENGTH = 100

BASE = Path.cwd().parent

RAW_DIR = BASE / "data/raw"
TRIM_DIR = BASE / "data/trimmed"
ASM_DIR = BASE / "data/assembly"
PHABOX_DIR = BASE / "data/phabox"
LOG_DIR = BASE / "logs"
RESULT_DIR = BASE / "results"

for d in [RAW_DIR, TRIM_DIR, ASM_DIR, PHABOX_DIR, LOG_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Directories created")

In [ ]:
#============================================
#3 Helper Function
#============================================

from IPython.display import display, Markdown


def run_cmd(cmd, step_name=None):
    print("\n" + "="*70)
    if step_name:
        print(f"STEP: {step_name}")
    print("="*70)
    print(cmd)
    print()

    process = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        executable="/bin/bash"
    )

    for line in process.stdout:
        print(line, end="")

    process.wait()

    if process.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

    print("\nDONE")

In [ ]:
#============================================
#4 Check Dependencies
#============================================

TOOLS = [
    "fastqc",
    "fastp",
    "megahit",
    "multiqc",
    "seqkit",
    "wget"
]

for tool in TOOLS:
    path = shutil.which(tool)
    print(f"{tool}: {'FOUND' if path else 'NOT FOUND'}")

In [ ]:
#============================================
#5 Disk Space Check
#============================================

usage = shutil.disk_usage("/")

print(f"Total: {usage.total / 1e9:.2f} GB")
print(f"Used : {usage.used / 1e9:.2f} GB")
print(f"Free : {usage.free / 1e9:.2f} GB")

In [ ]:
#============================================
#6 Download FASTQ from ENA
#============================================

# ENA direct download links
# Replace if ENA changes mirror/path

R1_URL = "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR376/093/SRR37631993/SRR37631993_1.fastq.gz"
R2_URL = "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR376/093/SRR37631993/SRR37631993_2.fastq.gz"

run_cmd(
    f'''wget -c -P {RAW_DIR} {R1_URL}''',
    "Download R1"
)

run_cmd(
    f'''wget -c -P {RAW_DIR} {R2_URL}''',
    "Download R2"
)

In [ ]:
#============================================
#7 Verify Downloaded FASTQ
#============================================

for f in RAW_DIR.glob("*.fastq.gz"):
    print(f"FOUND: {f.name}")

for f in RAW_DIR.glob("*.fastq.gz"):
    size_gb = f.stat().st_size / 1e9
    print(f"{f.name}: {size_gb:.2f} GB")

In [ ]:
#============================================
#8 Trimming + QC + Filtering Using fastp
#============================================

R1 = RAW_DIR / f"{SRA_ID}_1.fastq.gz"
R2 = RAW_DIR / f"{SRA_ID}_2.fastq.gz"

TRIM_R1 = TRIM_DIR / f"{SRA_ID}_R1.trimmed.fastq.gz"
TRIM_R2 = TRIM_DIR / f"{SRA_ID}_R2.trimmed.fastq.gz"

run_cmd(
    f'''fastp \
    --in1 {R1} \
    --in2 {R2} \
    --out1 {TRIM_R1} \
    --out2 {TRIM_R2} \
    --thread {THREADS} \
    --detect_adapter_for_pe \
    --length_required {MIN_LENGTH} \
    --compression 6 \
    --html {LOG_DIR}/fastp.html \
    --json {LOG_DIR}/fastp.json''',
    "fastp trimming"
)

In [ ]:
#============================================
#9 Remove Raw FASTQ
#============================================

#Optional only to save more disk space

for f in RAW_DIR.glob("*.fastq.gz"):
    os.remove(f)

print("Raw FASTQ removed to save space")

In [ ]:
#============================================
#10 Optional FastQC
#============================================

run_cmd(
    f'''fastqc \
    {TRIM_DIR}/*.fastq.gz \
    --threads {THREADS} \
    --outdir {LOG_DIR}''',
    "Trimmed FASTQC"
)

In [ ]:
#============================================
#11 Optional MultiQC
#============================================

run_cmd(
    f'''multiqc \
    {LOG_DIR} \
    -o {RESULT_DIR}''',
    "MultiQC"
)

In [ ]:
#============================================
#12 Assembly with MEGAHIT
#============================================

run_cmd(
    f'''megahit \
    -1 {TRIM_R1} \
    -2 {TRIM_R2} \
    -o {ASM_DIR} \
    -t {THREADS} \
    --min-contig-len 1000 \
    --out-prefix assembly''',
    "MEGAHIT Assembly"
)

In [ ]:
#============================================
#13 Assembly Summary
#============================================

CONTIGS = ASM_DIR / "assembly.contigs.fa"

run_cmd(
    f"seqkit stats {CONTIGS}",
    "Assembly Statistics"
)

In [ ]:
#============================================
#14 PhaBOX
#============================================

print("Assembly file ready for upload:")
print(CONTIGS)

"""
UPLOAD:
assembly.contigs.fa

TO:
https://phage.ee.cityu.edu.hk/phabox

SELECT:
- Virus identification
- Taxonomy
- Host prediction

OPTIONAL:
- CheckV
- iPHoP
- vConTACT2
"""